# Datathon - Case Passos Mágicos
## Análise PEDE 2022-2024 e previsão de risco de defasagem

Objetivo: analisar os indicadores educacionais e construir um modelo que estime a probabilidade de um aluno entrar em risco de defasagem.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay

ARQUIVO = "../data/BASE_DE_DADOS_PEDE_2024_DATATHON.xlsx"
xls = pd.ExcelFile(ARQUIVO)
xls.sheet_names


## 1. Limpeza e padronização

Os três anos possuem pequenas diferenças de nomes de colunas. Indicadores fora da escala esperada 0-10 são tratados como ausentes para evitar códigos auxiliares como 83 contaminando a análise.


In [ ]:
def preparar(aba, ano):
    df = pd.read_excel(ARQUIVO, sheet_name=aba)
    def col(*nomes):
        for n in nomes:
            if n in df.columns: return n
        return None
    out = pd.DataFrame({
        "RA": df["RA"].astype(str),
        "ano": ano,
        "fase": df["Fase"],
        "IAN": pd.to_numeric(df[col("IAN")], errors="coerce"),
        "IDA": pd.to_numeric(df[col("IDA")], errors="coerce"),
        "IEG": pd.to_numeric(df[col("IEG")], errors="coerce"),
        "IAA": pd.to_numeric(df[col("IAA")], errors="coerce"),
        "IPS": pd.to_numeric(df[col("IPS")], errors="coerce"),
        "IPP": pd.to_numeric(df[col("IPP")], errors="coerce") if col("IPP") else np.nan,
        "IPV": pd.to_numeric(df[col("IPV")], errors="coerce"),
        "INDE": pd.to_numeric(df[col(f"INDE {ano}", f"INDE {str(ano)[-2:]}")], errors="coerce"),
        "defasagem": pd.to_numeric(df[col("Defas","Defasagem")], errors="coerce")
    })
    for c in ["IAN","IDA","IEG","IAA","IPS","IPP","IPV","INDE"]:
        out.loc[~out[c].between(0,10.1),c] = np.nan
    return out

d22=preparar("PEDE2022",2022)
d23=preparar("PEDE2023",2023)
d24=preparar("PEDE2024",2024)
dados=pd.concat([d22,d23,d24],ignore_index=True)
dados.groupby("ano")[["INDE","IAN","IDA","IEG","IAA","IPS","IPP","IPV"]].mean().round(3)


## 2. Perfil de defasagem (IAN / Defasagem)

Para a análise de risco, considera-se defasagem quando o campo `Defas/Defasagem` é negativo. `-1` é apresentado como defasagem moderada e valores menores que `-1` como severa.


In [ ]:
def classe_defas(x):
    if pd.isna(x): return "Sem informação"
    if x >= 0: return "Adequado"
    if x == -1: return "Moderada"
    return "Severa"

dados["classe_defasagem"]=dados["defasagem"].apply(classe_defas)
perfil=pd.crosstab(dados["ano"],dados["classe_defasagem"],normalize="index")*100
perfil.round(1)


## 3. Relações entre os indicadores


In [ ]:
for ano,df in [(2022,d22),(2023,d23),(2024,d24)]:
    print("\nANO",ano)
    print(df[["IAN","IDA","IEG","IAA","IPS","IPP","IPV","INDE"]].corr().round(3)[["IDA","IPV","INDE"]])


## 4. Modelo longitudinal de risco

A modelagem usa dados do ano t para prever defasagem no ano t+1.  
Treino: 2022 -> 2023.  
Teste temporal: 2023 -> 2024.

O IPP não é utilizado porque não existe em 2022, evitando perda completa do primeiro período de treino.


In [ ]:
features=["IAN","IDA","IEG","IAA","IPS","IPV","INDE"]

def transicao(atual, seguinte):
    b=seguinte[["RA","defasagem"]].rename(columns={"defasagem":"defasagem_prox"})
    m=atual.merge(b,on="RA",how="inner")
    m["risco_proximo_ano"]=(m["defasagem_prox"]<0).astype(int)
    return m

treino=transicao(d22,d23)
teste=transicao(d23,d24)

X_train,y_train=treino[features],treino["risco_proximo_ano"]
X_test,y_test=teste[features],teste["risco_proximo_ano"]

modelo=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("model",RandomForestClassifier(
        n_estimators=400,max_depth=6,min_samples_leaf=8,
        class_weight="balanced",random_state=42
    ))
])
modelo.fit(X_train,y_train)
pred=modelo.predict(X_test)
prob=modelo.predict_proba(X_test)[:,1]

print(classification_report(y_test,pred,digits=3))
print("ROC-AUC:",round(roc_auc_score(y_test,prob),3))
print("Matriz de confusão:\n",confusion_matrix(y_test,pred))


## 5. Importância das variáveis


In [ ]:
importancia=pd.Series(modelo.named_steps["model"].feature_importances_,index=features).sort_values(ascending=False)
display(importancia)
importancia.sort_values().plot(kind="barh",title="Importância das variáveis - Random Forest")
plt.xlabel("Importância")
plt.show()


## Conclusão

O modelo deve funcionar como alerta de priorização, e não como decisão automática. Os resultados devem ser combinados com acompanhamento pedagógico, psicopedagógico e psicossocial da Passos Mágicos.
